In [0]:
from pyspark.sql.functions import regexp_extract, col, trim

# 1. Load the silver table
df_schedules = spark.table("railway_analytics.silver.fact_train_schedules")

# 2. Extract the Station Code using Regex
# This looks for the text after the " - " at the end of the string
df_schedules_cleaned = df_schedules.withColumn(
    "stn_code", 
    regexp_extract(col("station_name_raw"), r"- ([A-Z0-9]+)$", 1)
).withColumn(
    "clean_station_name", 
    trim(regexp_extract(col("station_name_raw"), r"^(.*?) -", 1))
)

# 3. Overwrite the table with the clean version
df_schedules_cleaned.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("railway_analytics.silver.fact_train_schedules")

print("Station codes extracted successfully!")

Station codes extracted successfully!


In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. Load the 76.8M raw records from the Bronze table
df_bronze = spark.table("railway_analytics.bronze.raw_train_delays")

# 2. This solves the '00961' vs '961' mismatch and joins
df_silver_base = df_bronze.withColumn("train_no", col("train_no").cast("int")).withColumn("last_processed_at", current_timestamp())

# 3. Write to the Silver Layer in Unity Catalog
# Using Delta format ensures ACID compliance 
(df_silver_base.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("railway_analytics.silver.fact_train_delays_base"))

print("Table 'fact_train_delays_base' created and optimized for joining.")

Table 'fact_train_delays_base' created and optimized for joining.


In [0]:
# 1. Load the tables
fact_delays = spark.table("railway_analytics.silver.fact_train_delays_base")
dim_schedules = spark.table("railway_analytics.silver.fact_train_schedules")

# 2.Join: Using multiple keys to prevent row explosion
# We join on both train number and station name
df_final_enriched = fact_delays.join(
    dim_schedules, 
    (fact_delays.train_no == dim_schedules.train_no) & 
    (fact_delays.station_name == dim_schedules.clean_station_name), 
    how="inner"
)
# # Checking a sample to verify 961 matches 961
# display(df_final_enriched.filter(col("train_no") == 961).limit(5))

# df_final_enriched.count()


# 3. Check the count again
final_count = df_final_enriched.count()
print(f"Original Count: 76,857,406")
print(f"Enriched Count: {final_count}")



Original Count: 76,857,406
Enriched Count: 610978


In [0]:
from pyspark.sql.functions import trim, col, broadcast

# 1. Load the tables
fact_delays = spark.table("railway_analytics.silver.fact_train_delays_base")
dim_schedules = spark.table("railway_analytics.silver.fact_train_schedules")

# 2. The Correct Join: Matching Train Number AND Station CODE since my old join was joining the wrong columns
# trim() just in case there are invisible spaces
df_final_enriched = fact_delays.join(
    broadcast(dim_schedules), 
    (fact_delays.train_no == dim_schedules.train_no) & 
    (trim(fact_delays.station_name) == trim(dim_schedules.stn_code)), 
    how="inner"
).drop(dim_schedules.train_no)

# 3. Just renaming it for no confusion
df_final_enriched = df_final_enriched.withColumnRenamed("station_name", "station_code")

# 4. Check the massive recovery!
print(f"Recovered Enriched Count: {df_final_enriched.count()}")

Recovered Enriched Count: 25844456


In [0]:

(df_final_enriched.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("railway_analytics.silver.fact_train_delays_enriched"))

print("Table successfully saved without duplicate columns!")

Table successfully saved without duplicate columns!
